# 用 HDFStore 和 PyTables 学习 tables 库

这份 notebook 根据原文整理，目标是用更小的数据量跑通核心流程：

1. 生成一个模拟金融时间序列数据集  
2. 用 `pandas.HDFStore` 保存和读取  
3. 用 `format='table'` 的方式写入，体验可追加、可查询的表格式  
4. 用 `tables`（PyTables）直接查看 HDF5 文件内部结构

为了更适合学习和快速运行，这里把原文里的超大数据量改小了。

## 1. 准备数据

下面这个函数模仿原文里的 `generate_sample_data()`：  
- 索引是分钟级时间序列  
- 每一列都是围绕 100 上下波动的浮点数  
- 用随机游走制造一点“像金融数据”的感觉

In [1]:
import os
import tempfile

import numpy as np
import pandas as pd

def generate_sample_data(rows=10_000, cols=5, seed=42):
    """Generate a reproducible financial-like DataFrame."""
    rng = np.random.default_rng(seed)
    index = pd.date_range("2021-01-01", periods=rows, freq="min")

    steps = rng.normal(loc=0.0, scale=0.02, size=(rows, cols))
    data = 100 + np.cumsum(steps, axis=0)
    columns = [f"No{i}" for i in range(cols)]
    return pd.DataFrame(data, index=index, columns=columns).round(4)

data = generate_sample_data(rows=10_000, cols=5)
data.head()

,No0,No1,No2,No3,No4
2021-01-01 00:00:00,100.0061,99.9792,100.0150,100.0188,99.9610
2021-01-01 00:01:00,99.9801,99.9818,100.0087,100.0185,99.9439
2021-01-01 00:02:00,99.9976,99.9973,100.0100,100.0410,99.9533
2021-01-01 00:03:00,99.9805,100.0047,99.9908,100.0586,99.9523
2021-01-01 00:04:00,99.9768,99.9911,100.0153,100.0555,99.9437


## 2. 查看数据规模

原文使用的是 500 万行、10 列。  
这里先缩小到 1 万行、5 列，便于本地快速体验。

In [2]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 10000 entries, 2021-01-01 00:00:00 to 2021-01-07 22:39:00
Freq: min
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   No0     10000 non-null  float64
 1   No1     10000 non-null  float64
 2   No2     10000 non-null  float64
 3   No3     10000 non-null  float64
 4   No4     10000 non-null  float64
dtypes: float64(5)
memory usage: 468.8 KB


## 3. 使用 `HDFStore` 直接保存整个 DataFrame

这是最直接的方式。  
优点是简单、快；缺点是灵活性相对少一些。

In [3]:
tmp_dir = tempfile.mkdtemp(prefix="hdfstore_demo_")
h5_path = os.path.join(tmp_dir, "data.h5")

h5 = pd.HDFStore(h5_path, mode="w")
h5["data"] = data
h5.close()

file_size_mb = os.path.getsize(h5_path) / (1024 * 1024)
file_size_mb

0.46466827392578125

In [4]:
h5 = pd.HDFStore(h5_path, mode="r")
data_copy = h5["data"]
h5.close()

data_copy.head()

,No0,No1,No2,No3,No4
2021-01-01 00:00:00,100.0061,99.9792,100.0150,100.0188,99.9610
2021-01-01 00:01:00,99.9801,99.9818,100.0087,100.0185,99.9439
2021-01-01 00:02:00,99.9976,99.9973,100.0100,100.0410,99.9533
2021-01-01 00:03:00,99.9805,100.0047,99.9908,100.0586,99.9523
2021-01-01 00:04:00,99.9768,99.9911,100.0153,100.0555,99.9437


In [5]:
data_copy.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 10000 entries, 2021-01-01 00:00:00 to 2021-01-07 22:39:00
Freq: min
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   No0     10000 non-null  float64
 1   No1     10000 non-null  float64
 2   No2     10000 non-null  float64
 3   No3     10000 non-null  float64
 4   No4     10000 non-null  float64
dtypes: float64(5)
memory usage: 468.8 KB


## 4. 使用 `format='table'` 写入

`table` 格式更灵活，支持追加和按条件查询。  
代价是写入和读取会稍慢一点，文件也可能略大。

In [6]:
data.to_hdf(h5_path, key="data_table", format="table", mode="w")

file_size_table_mb = os.path.getsize(h5_path) / (1024 * 1024)
file_size_table_mb

0.5330562591552734

In [7]:
data_table_copy = pd.read_hdf(h5_path, key="data_table")
data_table_copy.head()

,No0,No1,No2,No3,No4
2021-01-01 00:00:00,100.0061,99.9792,100.0150,100.0188,99.9610
2021-01-01 00:01:00,99.9801,99.9818,100.0087,100.0185,99.9439
2021-01-01 00:02:00,99.9976,99.9973,100.0100,100.0410,99.9533
2021-01-01 00:03:00,99.9805,100.0047,99.9908,100.0586,99.9523
2021-01-01 00:04:00,99.9768,99.9911,100.0153,100.0555,99.9437


## 5. 直接用 PyTables 查看 HDF5 结构

这里开始接触 `tables` 库本身。  
`pandas` 的 `HDFStore` 底层就是借助它来处理 HDF5 文件的。

In [9]:
import tables as tb

h5 = tb.open_file(h5_path, mode="r")

In [10]:
print(h5)

/tmp/hdfstore_demo_s2ljpqka/data.h5 (File) np.str_('')
Last modif.: '2026-04-20T07:29:01+00:00'
Object Tree: 
/ (RootGroup) np.str_('')
/data_table (Group) np.str_('')
/data_table/table (Table(np.int64(10000),)) np.str_('')



In [11]:
print(h5.root.data_table)

/data_table (Group) np.str_('')


In [12]:
print(h5.root.data_table.table[:3])

[(1609459200000000000, [100.0061,  99.9792, 100.015 , 100.0188,  99.961 ])
 (1609459260000000000, [ 99.9801,  99.9818, 100.0087, 100.0185,  99.9439])
 (1609459320000000000, [ 99.9976,  99.9973, 100.01  , 100.041 ,  99.9533])]


In [13]:
h5.close()

## 6. 清理文件

程序运行结束记得把临时文件删掉。  

In [14]:
os.remove(h5_path)
os.rmdir(tmp_dir)

## 总结

- `HDFStore` 适合把整个 `DataFrame` 快速存进 HDF5 文件  
- `format='table'` 更灵活，可以追加和查询  
- `tables` 可以直接查看 HDF5 内部结构  